# 13.3 · GAN 基础 / Generative Adversarial Networks

> **课程定位 / Where this fits**
> 第 3 课，**Part 13 · 生成模型**。一条与 VAE 完全不同的生成路线, 曾是图像生成的霸主。
> Lesson 3, **Part 13 · Generative Models**. A route totally different from VAE, once the king of image generation.
>
> VAE 通过"建模分布 + 重建"来生成, 结果偏模糊。**GAN(生成对抗网络)** 用一个绝妙的想法：**让两个网络互相博弈**——**生成器(Generator)** 努力造假图骗过对手, **判别器(Discriminator)** 努力分辨真假。两者在对抗中"军备竞赛", 最终生成器学会造出**以假乱真、远比 VAE 锐利**的图。但 GAN **训练出了名地不稳定**(模式坍塌、不收敛)。本课从零搭 GAN、在 MNIST 上训练、**看数字从噪声中逐渐浮现**, 并讲清 minimax 目标与模式坍塌等经典难题。
> VAE generates via "model the distribution + reconstruct," yielding blurry results. The **GAN** uses a brilliant idea: **two networks compete** — a **Generator** tries to fake images to fool an opponent, a **Discriminator** tries to tell real from fake. Through this arms race the generator learns to produce images **far sharper than VAE**. But GANs are **notoriously unstable to train** (mode collapse, non-convergence). We build a GAN from scratch, train on MNIST, **watch digits emerge from noise**, and explain the minimax objective and classic pitfalls.
>
> 💼 **实战/面试视角**："GAN 的 minimax 目标 / 生成器判别器怎么交替训练 / 模式坍塌 / GAN vs VAE / 训练不稳定的原因" 是生成模型高频。
> 💼 **Practical/interview angle:** "GAN minimax objective / alternating training / mode collapse / GAN vs VAE / why unstable" — high-frequency.

> 📐 **符号约定 / Notation**
> - $G$ —— 生成器: 噪声 $z$ → 假图 / generator: noise → fake image
> - $D$ —— 判别器: 图 → 真/假概率 / discriminator: image → real/fake probability
> - minimax —— $G$ 最小化、$D$ 最大化的对抗目标 / the adversarial objective

> 💡 **面试相关 / Interview-relevant**
> - "GAN 的两个网络与对抗目标"（出镜率 ★★★★★）
> - "生成器和判别器如何交替训练"（★★★★★）
> - "模式坍塌(mode collapse)是什么/为什么"（★★★★★）
> - "GAN vs VAE(锐利vs模糊, 不稳vs稳)"（★★★★）
> - "非饱和损失/为什么G用 log D(G(z))"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 GAN 的对抗博弈思想与 minimax 目标。
   Understand GAN's adversarial game and minimax objective.
2. 掌握生成器/判别器的**交替训练**流程。
   Master the alternating training of generator/discriminator.
3. **从零搭 GAN** 训练, 看数字从噪声浮现。
   Build a GAN from scratch, watch digits emerge from noise.
4. 理解**模式坍塌**与训练不稳定。
   Understand mode collapse and training instability.

## 目录 / TOC
1. [对抗博弈：造假者 vs 鉴定者 ⭐](#1)
2. [minimax 目标与交替训练 ⭐](#2)
3. [从零搭 GAN 训练：数字浮现 ⭐](#3)
4. [模式坍塌、不稳定 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 对抗博弈：造假者 vs 鉴定者 ⭐ / The Adversarial Game

GAN 的核心是一个**对抗博弈**, 经典比喻：
GAN's core is an **adversarial game**, the classic analogy:
- **生成器 $G$ = 造假币的人**：输入一串随机噪声 $z$, 输出一张假图。目标是造得**越像真的越好**, 骗过鉴定者。
  **Generator $G$ = counterfeiter:** takes random noise $z$, outputs a fake image. Goal: make it **look as real as possible** to fool the detective.
- **判别器 $D$ = 鉴定者(警察)**：输入一张图(可能真可能假), 输出"它是真图的概率"。目标是**准确分辨真假**。
  **Discriminator $D$ = detective:** takes an image (real or fake), outputs "probability it's real." Goal: **tell real from fake accurately**.

两者**同时训练、互相对抗**：判别器越来越会鉴别, 逼着生成器造得越来越逼真；生成器越来越会造假, 逼着判别器越来越精明。这场"军备竞赛"的**纳什均衡**是：生成器造出的假图和真图分布**一模一样**, 判别器再也分不出(只能瞎猜 50%)。此时生成器就学会了真实数据分布——**能生成以假乱真的新图**。
They **train together, against each other:** the discriminator gets better at detecting, forcing the generator to fake more convincingly; the generator gets better, forcing the discriminator sharper. The **Nash equilibrium** of this arms race: the generator's fakes match the real distribution **exactly**, and the discriminator can't tell them apart (50% guessing). At that point the generator has learned the real data distribution — **generating convincing new images**.

**关键**：判别器其实是给生成器提供了一个**可学习的、自适应的损失函数**——不用人手工定义"什么叫像真数字", 而是让判别器自己学出来, 再反过来指导生成器。这是 GAN 最深刻的地方。
**Key:** the discriminator effectively provides the generator with a **learned, adaptive loss function** — instead of hand-defining "what looks like a real digit," the discriminator learns it and guides the generator. GAN's deepest idea.


<a id="2"></a>
## 2. minimax 目标与交替训练 ⭐ / Minimax Objective & Alternating Training

数学上是一个 **minimax(极小极大)博弈**：
Mathematically a **minimax game**:

$$\min_G \max_D \; \mathbb{E}_{x\sim\text{真实}}[\log D(x)] + \mathbb{E}_{z\sim N(0,1)}[\log(1 - D(G(z)))]$$

- **$D$ 想最大化**：把真图判为真($D(x)\to1$)、把假图判为假($D(G(z))\to0$)。
  **$D$ maximizes:** real → 1, fake → 0.
- **$G$ 想最小化**：让假图被判为真($D(G(z))\to1$), 即骗过 $D$。
  **$G$ minimizes:** make fakes judged real (fool $D$).

**怎么训练这个博弈**(面试核心)：**交替**更新两个网络——
**How to train this game** (interview core): **alternate** updating the two networks —
1. **训练 $D$**：固定 $G$, 喂一批真图(标签=真) + 一批 $G$ 造的假图(标签=假), 让 $D$ 学会分辨。
   **Train $D$:** freeze $G$; feed real images (label real) + $G$'s fakes (label fake); $D$ learns to discriminate.
2. **训练 $G$**：固定 $D$, 让 $G$ 造图喂给 $D$, **目标是让 $D$ 判它为真**(用 $D$ 的梯度更新 $G$)。
   **Train $G$:** freeze $D$; $G$'s fakes go to $D$, **aiming for $D$ to call them real** (update $G$ via $D$'s gradient).

> **非饱和损失(non-saturating)**(面试点)：实际训 $G$ 不最小化 $\log(1-D(G(z)))$(早期 $D$ 太强时梯度消失), 而是**最大化 $\log D(G(z))$**——同样的方向但梯度更健康。本课用它。
> **Non-saturating loss:** in practice $G$ doesn't minimize $\log(1-D(G(z)))$ (gradients vanish early when $D$ is strong); instead it **maximizes $\log D(G(z))$** — same direction, healthier gradients. We use this.


<a id="3"></a>
## 3. 从零搭 GAN 训练：数字浮现 ⭐ / Build & Train: Digits Emerge

从零搭一个 MLP GAN 在 MNIST 上训练。我们在训练过程中**定期保存生成结果**, 看数字如何从纯噪声里逐渐"浮现"出来。
Build an MLP GAN from scratch on MNIST. We **snapshot generations periodically** during training to watch digits "emerge" from pure noise.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white"); torch.manual_seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
# 归一化到 [-1,1] 配合生成器的 Tanh 输出 / normalize to [-1,1] to match Tanh output
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
loader = DataLoader(Subset(datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tfm), range(15000)),
                    batch_size=128, shuffle=True, drop_last=True)
Z = 64
G = nn.Sequential(nn.Linear(Z,256), nn.LeakyReLU(0.2), nn.Linear(256,512), nn.LeakyReLU(0.2),
                  nn.Linear(512,784), nn.Tanh())                              # 噪声→假图(Tanh→[-1,1]) / noise→fake
D = nn.Sequential(nn.Flatten(), nn.Linear(784,512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
                  nn.Linear(512,256), nn.LeakyReLU(0.2), nn.Linear(256,1))    # 图→真/假logit / image→real/fake
bce = nn.BCEWithLogitsLoss()
og = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5,0.999))                # GAN 常用 betas=(0.5,0.999) / common betas
od = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5,0.999))
fixed_z = torch.randn(8, Z)                                                    # 固定噪声, 看同一批"种子"的演变 / fixed seeds
snapshots = {}
t0 = time.time()
for ep in range(40):
    for xb, _ in loader:
        b = xb.size(0); z = torch.randn(b, Z); fake = G(z).view(-1,1,28,28)
        # --- 训 D: 真判真, 假判假 / train D: real→real, fake→fake ---
        od.zero_grad()
        loss_d = bce(D(xb), torch.ones(b,1)*0.9) + bce(D(fake.detach()), torch.zeros(b,1))  # 0.9=标签平滑 / label smoothing
        loss_d.backward(); od.step()
        # --- 训 G: 让 D 把假图判为真(非饱和损失) / train G: fool D (non-saturating) ---
        og.zero_grad()
        loss_g = bce(D(fake), torch.ones(b,1))            # 最大化 log D(G(z)) / maximize log D(G(z))
        loss_g.backward(); og.step()
    if ep in (0, 5, 15, 39):
        with torch.no_grad(): snapshots[ep] = G(fixed_z).view(-1,28,28)
print(f"GAN 训练完成 ({time.time()-t0:.0f}s), D损失={loss_d.item():.2f}, G损失={loss_g.item():.2f}")
# 看同一批噪声种子在不同epoch生成的图: 从噪声→数字 / evolution of the same seeds across epochs
fig, axes = plt.subplots(len(snapshots), 8, figsize=(11, 1.4*len(snapshots)))
for r,(ep,imgs) in enumerate(snapshots.items()):
    for c in range(8):
        axes[r,c].imshow(imgs[c], cmap="gray"); axes[r,c].axis("off")
    axes[r,0].set_ylabel(f"ep{ep}", fontsize=9, rotation=0, labelpad=18)
fig.suptitle("GAN 训练演变(同一批噪声种子): 从纯噪声 → 逐渐浮现出数字"); plt.tight_layout(); plt.show()
print("早期(ep0)是纯噪声; 随训练, 生成器在判别器的'对抗压力'下逐渐学会造出数字形状")


<a id="4"></a>
## 4. 模式坍塌、不稳定 + 小结 ⭐ / Mode Collapse & Instability

GAN 效果惊艳(比 VAE 锐利), 但**训练出了名地难**——几个经典难题(面试必问)：
GANs are striking (sharper than VAE) but **notoriously hard to train** — classic pitfalls (interview):
- **模式坍塌(mode collapse)**：生成器找到"一个能骗过判别器的样本", 就**反复只生成它(或少数几种)**, 丧失多样性。比如只会生成数字"1", 不管输入什么噪声。原因: 生成器只需骗过当前判别器, 没有动力覆盖所有模式。
  **Mode collapse:** the generator finds "one sample that fools $D$" and **only produces it (or a few)**, losing diversity — e.g. only generating "1" regardless of noise. It only needs to fool the current $D$, with no incentive to cover all modes.
- **训练不稳定/不收敛**：两个网络此消彼长, 损失可能剧烈震荡而非收敛(我们看到的 D/G 损失拉锯就是)。判别器太强→生成器梯度消失；太弱→给不了有用信号。**需要精心平衡**。
  **Instability/non-convergence:** the two networks seesaw; losses may oscillate rather than converge (the D/G tug-of-war we saw). $D$ too strong → $G$'s gradients vanish; too weak → no useful signal. **Delicate balance needed.**

下面检查我们的生成器**有没有模式坍塌**(看生成样本是否多样、覆盖多种数字)。
Let's check whether our generator **mode-collapsed** (are samples diverse, covering different digits).


In [ ]:
# 生成一批样本, 看多样性(是否覆盖多种数字 = 没坍塌) / generate a batch, check diversity
with torch.no_grad(): samples = G(torch.randn(40, Z)).view(-1,28,28)
fig, axes = plt.subplots(4, 10, figsize=(12, 5))
for i in range(40): axes[i//10,i%10].imshow(samples[i], cmap="gray"); axes[i//10,i%10].axis("off")
fig.suptitle("GAN 生成的 40 个样本: 检查多样性(覆盖多种数字=健康; 全都一样=模式坍塌)"); plt.tight_layout(); plt.show()
diversity = samples.std(0).mean().item()                  # 样本间逐像素标准差(多样性代理) / diversity proxy
print(f"样本多样性(逐像素标准差均值) = {diversity:.3f}  (越大越多样; 接近0=模式坍塌只生成同一张)")
print("我们的GAN生成了多种不同数字 → 没有严重模式坍塌(但小MLP/CPU下数字偏粗糙)")
print("\n💡 稳定GAN训练的常用技巧(下一课展开): 卷积结构(DCGAN) / Wasserstein损失(WGAN) / 谱归一化 / 标签平滑 / 平衡D和G")


```
GAN: 生成器G(噪声→假图) vs 判别器D(图→真假概率) 对抗博弈; D提供'可学习的损失'指导G
minimax: min_G max_D E[logD(真)] + E[log(1-D(G(z)))]; D想分清, G想骗过
交替训练: ①固定G训D(真判真假判假) ②固定D训G(让假图被判真); 非饱和损失 G 最大化 logD(G(z))
均衡: G造的假图分布=真实分布, D只能瞎猜50% → G学会数据分布
模式坍塌: G只生成少数几种能骗过D的样本, 丧失多样性(经典难题)
训练不稳定: 两网络拉锯, 损失震荡; D太强G梯度消失, D太弱无信号; 需精心平衡
GAN vs VAE: GAN锐利但难训/无显式似然; VAE模糊但稳/有隐空间
```

### 💡 面试速查 / Interview cheat-sheet
1. **GAN结构**: G(噪声→假图) + D(判真假); 对抗博弈; D=可学习的损失。
   GAN: G(noise→fake) + D(real/fake); adversarial game; D = learned loss.
2. **minimax**: D最大化分辨, G最小化(骗过D); 交替训练。
   Minimax: D maximizes discrimination, G minimizes (fools D); alternate.
3. **模式坍塌**: G只生成少数样本骗过D, 丧失多样性。
   Mode collapse: G produces few samples that fool D, losing diversity.
4. **不稳定**: 两网络拉锯, 损失震荡; 需平衡D/G。
   Instability: seesaw, oscillating loss; must balance D/G.
5. **GAN vs VAE**: 锐利vs模糊, 难训vs稳定, 无似然vs有隐空间。
   GAN vs VAE: sharp vs blurry, hard vs stable, no likelihood vs latent space.

### 下一节 / Next
**13.4 DCGAN / WGAN**——针对 GAN 训练不稳定, 两个关键改进: **DCGAN** 用卷积结构生成更好的图; **WGAN** 改用 **Wasserstein 距离**作损失, 大幅稳定训练、缓解模式坍塌。我们会理解这些改进为何有效。
**13.4 DCGAN / WGAN** — two key fixes for GAN instability: **DCGAN** uses convolutional architecture for better images; **WGAN** replaces the loss with **Wasserstein distance**, greatly stabilizing training and easing mode collapse. We'll understand why these work.
